# 03. 실험 1~5 (변수 조합 비교)

가이드 5장의 실험 1~5를 자동 루프로 실행한다.

**사용 방법:**
1. 아래 "변수 슬롯"에서 자기 변수를 그룹별 리스트에 추가/주석처리
2. `experiments` 사전에 자기 실험 조합 추가
3. 셀 전체 실행 → 결과 표 + Feature Importance 자동 출력

**변수 추가/제거 시 한 줄 주석으로 채택/탈락 표시.** 자세한 근거는 보고서에 작성.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import sys
import os

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 한글 폰트
import platform
if platform.system() == 'Darwin':
    matplotlib.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    matplotlib.rcParams['font.family'] = 'Malgun Gothic'
else:
    matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 재현성 고정
RANDOM_STATE = 42
TEST_SIZE = 0.2

## 1. 데이터 로드

변수가 모두 추가된 마스터 데이터셋을 로드한다.
(전처리/변수 생성은 01, 02 노트북에서 완료)

In [ ]:
df = pd.read_csv('../data/processed/apt_master.csv', encoding='utf-8-sig')

# 타겟 로그 변환
df['log_price'] = np.log1p(df['거래금액(만원)'])

print(f'데이터 shape: {df.shape}')
print(f'컬럼: {df.columns.tolist()}')

## 2. 변수 슬롯

각 그룹에 자기 변수를 추가/주석처리. 채택/탈락 한 줄 표시.
자세한 근거는 보고서에서 작성.

In [ ]:
# ==========================================================================
# 변수 슬롯 — 팀원이 자기 변수를 여기에 추가/주석처리
# ==========================================================================

PHYSICAL = [
    '전용면적(㎡)',
    '층',
    '노후도',
]

REGION = [
    '강남권',
    # '구',                    # 원핫인코딩 후 추가 검토
]

TRANSPORT = [
    # 'subway_nearest_dist',   # 예시 — 02 노트북에서 생성됐다면 주석 해제
    # 'bus_count_500m',        # 탈락 (효과 약함)
]

EDUCATION = [
    # 'school_count_1km',
    # 'academy_count_500m',
]

CONVENIENCE = [
    # 'mart_nearest_dist',
    # 'hospital_nearest_dist',
]

NEGATIVE = [
    # 'funeral_nearest_dist',
    # 'incinerator_nearest_dist',
]

TARGET = 'log_price'

# 결측치 확인
all_features = PHYSICAL + REGION + TRANSPORT + EDUCATION + CONVENIENCE + NEGATIVE
missing_cols = [c for c in all_features if c not in df.columns]
if missing_cols:
    print(f'⚠️  존재하지 않는 컬럼: {missing_cols}')
    print('   → 슬롯에서 주석처리하거나 02 노트북에서 변수 추가 필요')
else:
    print(f'✅ 모든 변수 존재 확인 ({len(all_features)}개)')

## 3. 실험 정의

가이드 5장의 실험 1~5를 사전으로 정의. 한 줄 추가/수정으로 자유롭게 실험 추가 가능.

In [ ]:
# ==========================================================================
# 실험 정의 — 변수 조합 자유롭게 추가/수정
# ==========================================================================

experiments = {
    '실험1 (기본)':         PHYSICAL + REGION,
    '실험2 (+ 교통)':       PHYSICAL + REGION + TRANSPORT,
    '실험3-A (+ 긍정)':     PHYSICAL + REGION + TRANSPORT + EDUCATION + CONVENIENCE,
    '실험3-B (+ 부정)':     PHYSICAL + REGION + TRANSPORT + NEGATIVE,
    '실험3-C (+ 전체)':     PHYSICAL + REGION + TRANSPORT + EDUCATION + CONVENIENCE + NEGATIVE,
    '실험4 (최종)':         PHYSICAL + REGION + TRANSPORT + EDUCATION + CONVENIENCE + NEGATIVE,
}

# 빈 변수 그룹은 자동 스킵 (실험2를 정의했지만 TRANSPORT가 비었으면 실험1과 같음)
experiments = {name: cols for name, cols in experiments.items() if cols}

print('정의된 실험:')
for name, cols in experiments.items():
    print(f'  {name}: 변수 {len(cols)}개')

## 4. 실험 실행 함수

In [ ]:
def run_experiment(df, feature_cols, target_col, exp_name,
                   test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """
    단일 실험 실행: Ridge + RandomForest 두 모델로 학습/평가.

    Returns
    -------
    list of dict : 각 모델별 결과
    """
    # 결측치 제거
    data = df.dropna(subset=feature_cols + [target_col]).copy()

    X = data[feature_cols]
    y = data[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    results = []

    # --- Ridge (스케일링 필요) ---
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    ridge = Ridge(alpha=1.0, random_state=random_state)
    ridge.fit(X_train_s, y_train)
    ridge_pred = ridge.predict(X_test_s)
    results.append({
        '실험': exp_name,
        '모델': 'Ridge',
        '변수수': len(feature_cols),
        'RMSE': np.sqrt(mean_squared_error(y_test, ridge_pred)),
        'R2': r2_score(y_test, ridge_pred),
        'n_samples': len(data),
    })

    # --- Random Forest (스케일링 불필요) ---
    rf = RandomForestRegressor(n_estimators=100, random_state=random_state, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    results.append({
        '실험': exp_name,
        '모델': 'RandomForest',
        '변수수': len(feature_cols),
        'RMSE': np.sqrt(mean_squared_error(y_test, rf_pred)),
        'R2': r2_score(y_test, rf_pred),
        'n_samples': len(data),
    })

    return results, rf  # rf는 Feature Importance용으로 반환


def run_all_experiments(df, experiments, target_col):
    """모든 실험을 순차 실행 후 결과 DataFrame 반환."""
    all_results = []
    last_rf_models = {}  # 실험별 마지막 RF 모델 저장 (Feature Importance용)

    for exp_name, feature_cols in experiments.items():
        print(f'\n▶ {exp_name} 실행 중...')
        results, rf_model = run_experiment(df, feature_cols, target_col, exp_name)
        all_results.extend(results)
        last_rf_models[exp_name] = (rf_model, feature_cols)
        for r in results:
            print(f"  [{r['모델']:>12}] RMSE: {r['RMSE']:.4f} | R²: {r['R2']:.4f}")

    return pd.DataFrame(all_results), last_rf_models

## 5. 실험 실행

In [ ]:
results_df, rf_models = run_all_experiments(df, experiments, TARGET)

print('\n=== 전체 결과 ===')
print(results_df.to_string(index=False))

## 6. 결과 비교 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# R² 비교
pivot_r2 = results_df.pivot(index='실험', columns='모델', values='R2')
pivot_r2.plot(kind='bar', ax=axes[0], color=['#2c7fb8', '#d95f02'])
axes[0].set_title('실험별 R² 비교')
axes[0].set_ylabel('R²')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(loc='lower right')
axes[0].grid(axis='y', alpha=0.3)

# RMSE 비교
pivot_rmse = results_df.pivot(index='실험', columns='모델', values='RMSE')
pivot_rmse.plot(kind='bar', ax=axes[1], color=['#2c7fb8', '#d95f02'])
axes[1].set_title('실험별 RMSE 비교 (낮을수록 좋음)')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(loc='upper right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 최종 모델 Feature Importance

가장 변수 많이 쓴 실험(보통 실험4)의 RandomForest Feature Importance.
변수 그룹별로도 합산해서 어떤 그룹이 가장 기여했는지 확인.

In [ ]:
# 최종 실험 선택 (변수 가장 많은 실험)
final_exp = max(experiments.items(), key=lambda x: len(x[1]))[0]
rf_model, feature_cols = rf_models[final_exp]

# 변수별 중요도
importance = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(15, max(5, len(feature_cols) * 0.3)))

# 개별 변수
importance.plot(kind='barh', ax=axes[0], color='#2c7fb8')
axes[0].set_title(f'Feature Importance — {final_exp}')
axes[0].set_xlabel('Importance')

# 그룹별 합산
group_map = {}
for c in PHYSICAL:    group_map[c] = '물리'
for c in REGION:      group_map[c] = '지역'
for c in TRANSPORT:   group_map[c] = '교통'
for c in EDUCATION:   group_map[c] = '교육'
for c in CONVENIENCE: group_map[c] = '편의'
for c in NEGATIVE:    group_map[c] = '환경(부정)'

group_importance = pd.Series({
    col: imp for col, imp in importance.items() if col in group_map
}).groupby(group_map).sum().sort_values()

group_importance.plot(kind='barh', ax=axes[1], color='#d95f02')
axes[1].set_title('변수 그룹별 중요도 합산')
axes[1].set_xlabel('Importance (합)')

plt.tight_layout()
plt.show()

In [ ]:
# === Ridge 계수 시각화 (선택) ===
# 변수의 가격 영향 방향 확인용
from sklearn.linear_model import Ridge

X = df.dropna(subset=feature_cols + [TARGET])[feature_cols]
y = df.dropna(subset=feature_cols + [TARGET])[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
ridge_final = Ridge(alpha=1.0).fit(X_train_s, y_train)

coef = pd.Series(ridge_final.coef_, index=feature_cols).sort_values()
colors = ['#d62728' if c < 0 else '#2ca02c' for c in coef]

plt.figure(figsize=(8, max(4, len(feature_cols) * 0.3)))
coef.plot(kind='barh', color=colors)
plt.title(f'Ridge 계수 (스케일링 후) — {final_exp}')
plt.xlabel('Coefficient')
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('양수 = 가격 상승 요인 / 음수 = 가격 하락 요인')

## 8. 실험 5: 강남권 / 비강남권 분리 학습

같은 변수로 두 지역을 따로 학습. 지역 특성에 따른 변수 작용 차이 확인.

In [ ]:
# 강남권 변수는 빼고 학습 (이미 그룹이 나뉘었으므로)
final_features = [c for c in experiments[final_exp] if c != '강남권']

gangnam_results = []
for region_name, region_df in [
    ('강남권', df[df['강남권'] == 1]),
    ('비강남권', df[df['강남권'] == 0]),
]:
    print(f'\n▶ {region_name} (n={len(region_df)}) 학습 중...')
    results, _ = run_experiment(region_df, final_features, TARGET, f'실험5-{region_name}')
    gangnam_results.extend(results)
    for r in results:
        print(f"  [{r['모델']:>12}] RMSE: {r['RMSE']:.4f} | R²: {r['R2']:.4f}")

gangnam_df = pd.DataFrame(gangnam_results)
print('\n=== 강남/비강남 비교 ===')
print(gangnam_df.to_string(index=False))

## 9. 결과 저장

실험 결과 CSV로 저장. 보고서 작성 시 참조.

In [ ]:
os.makedirs('../data/final', exist_ok=True)

results_df.to_csv('../data/final/experiment_results.csv',
                  index=False, encoding='utf-8-sig')
gangnam_df.to_csv('../data/final/experiment5_gangnam_split.csv',
                  index=False, encoding='utf-8-sig')

print('저장 완료:')
print('  - ../data/final/experiment_results.csv')
print('  - ../data/final/experiment5_gangnam_split.csv')